# Create dataset with CEOs of US Fortune 500 companies from 2018 to 2026

## Create dataset skeleton with [company name] [year] [CEO]

In [ ]:
import pandas as pd

rows = []

for year in range(2018, 2027):
    filename = f"data/fortune_500/fortune500_USA_{year}.csv"
    df = pd.read_csv(filename)

    for _, row in df.iterrows():
        rows.append({
            "year": year,
            "company_name": row["company"],
            "rank": row["rank"],
            "CEO": ""
        })

skeleton = pd.DataFrame(rows)
skeleton.to_csv("data/ceos.csv", index=False)

print(f"Saved {len(skeleton)} rows to data/ceos.csv")

Saved 4500 rows to data/ceos.csv


## Integrate data from other existing datasets

### 2023

In [ ]:
import pandas as pd
import json

# load the ceos.csv dataset
ceos = pd.read_csv("data/ceos.csv")
ceos["ceo"] = ceos["ceo"].astype(object)

# add the CEO_alt_name column if it doesn't exist yet
if "CEO_alt_name" not in ceos.columns:
    ceos["CEO_alt_name"] = ""
ceos["CEO_alt_name"] = ceos["CEO_alt_name"].astype(object)

# load the json file
with open("data/partial_ceos_data/ceos_usa_2023.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# build a dict: lowercase company name -> (ceo, ceo_alt)
ceo_lookup = {}
for entry in data:
    company = entry["Company"]
    ceo_lookup[company.lower()] = (entry["CEO"], entry["CEO_alt"])

# fill in ceos.csv for year 2023
mask_2023 = ceos["year"] == 2023

for idx in ceos[mask_2023].index:
    company = ceos.loc[idx, "company_name"]
    key = company.lower()
    if key in ceo_lookup:
        ceo_name, ceo_alt = ceo_lookup[key]
        ceos.loc[idx, "ceo"] = ceo_name
        ceos.loc[idx, "CEO_alt_name"] = ceo_alt
    else:
        rank = ceos.loc[idx, "rank"]
        print(f"company {company} (rank {rank}) couldn't be found in the file")

ceos.to_csv("data/ceos.csv", index=False)
print("done")

## Scrape remaining missing data from Wikipedia

### 2026 - URLs and CEOs

#### Get Wikipedia URLs and CEOs

In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Convert company name to Wikipedia URL format"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def extract_ceo_from_infobox(soup):
    """Extract CEO name(s) from Wikipedia infobox"""
    ceos = []
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    rows = infobox.find_all('tr')
    for row in rows:
        header = row.find('th')
        if header:
            header_text = header.get_text().strip().lower()
            if 'key people' in header_text or 'key person' in header_text:
                data = row.find('td')
                if data:
                    text = data.get_text()
                    lines = text.split('\n')
                    for line in lines:
                        if re.search(r'\b(CEO|Chief Executive Officer)\b', line, re.IGNORECASE):
                            name_match = re.match(r'^([^(\[]+)', line)
                            if name_match:
                                name = name_match.group(1).strip()
                                name = re.sub(r'\[.*?\]', '', name)
                                name = name.strip(' ,')
                                if name and len(name) > 2:
                                    ceos.append(name)
    return ', '.join(ceos) if ceos else None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

if "wikipedia_page" not in ceos.columns:
    ceos["wikipedia_page"] = ""
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index
total = len(indices)

for i, idx in enumerate(indices):
    company = ceos.loc[idx, "company"]
    company_alt = ceos.loc[idx, "company_alt_name"]
    print(f"\n[{i+1}/{total}] Processing: {company}")

    # build list of names to try: company name, then alt name if available
    names_to_try = [company]
    if pd.notna(company_alt) and str(company_alt).strip():
        names_to_try.append(str(company_alt).strip())

    ceo = None
    found_url = None
    for name in names_to_try:
        wiki_url = get_wikipedia_url(name)

        try:
            response = requests.get(wiki_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"  ✗ Request failed for '{name}': {str(e)[:80]}")
            continue

        if response.status_code == 404:
            print(f"  ✗ Wikipedia page not found for '{name}': {wiki_url}")
            continue
        elif response.status_code != 200:
            print(f"  ✗ HTTP {response.status_code} - page not accessible for '{name}'")
            continue

        try:
            soup = BeautifulSoup(response.content, 'html.parser')
            ceo = extract_ceo_from_infobox(soup)
        except Exception as e:
            print(f"  ✗ Unexpected error while parsing page for '{name}': {str(e)[:80]}")
            continue

        if ceo:
            found_url = wiki_url
            break
        else:
            print(f"  ✗ CEO not found in infobox for '{name}'")

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        ceos.loc[idx, "wikipedia_page"] = found_url
        print(f"  ✓ Found CEO: {ceo}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")


[1/500] Processing: Visa
  ✗ CEO not found in infobox for 'Visa'

[2/500] Processing: Travelers
  ✗ CEO not found in infobox for 'Travelers'

[3/500] Processing: Pfizer
  ✓ Found CEO: Albert Bourla

[4/500] Processing: Amazon
  ✗ CEO not found in infobox for 'Amazon'

[5/500] Processing: IBM
  ✓ Found CEO: Arvind Krishna

[6/500] Processing: Target
  ✗ CEO not found in infobox for 'Target'

[7/500] Processing: Bank of America
  ✓ Found CEO: Brian Moynihan

[8/500] Processing: Comcast
  ✓ Found CEO: Brian L. Roberts, Michael J. Cavanagh

[9/500] Processing: McKesson
  ✓ Found CEO: Brian S. Tyler

[10/500] Processing: Exelon
  ✓ Found CEO: Calvin Butler

[11/500] Processing: Wells Fargo
  ✓ Found CEO: Steven Black

[12/500] Processing: Costco Wholesale
  ✓ Found CEO: Hamilton E. James

[13/500] Processing: Honeywell International
  ✓ Found CEO: Darius Adamczyk

[14/500] Processing: Boeing
  ✓ Found CEO: Kelly Ortberg

[15/500] Processing: UnitedHealth Group
  ✓ Found CEO: Stephen J. Hem

#### Get CEOs of manually inserted URLs

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def extract_ceo_from_infobox(soup):
    """Extract CEO name(s) from Wikipedia infobox"""
    ceos = []
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    rows = infobox.find_all('tr')
    for row in rows:
        header = row.find('th')
        if header:
            header_text = header.get_text().strip().lower()
            if 'key people' in header_text or 'key person' in header_text:
                data = row.find('td')
                if data:
                    text = data.get_text()
                    lines = text.split('\n')
                    for line in lines:
                        if re.search(r'\b(CEO|Chief Executive Officer)\b', line, re.IGNORECASE):
                            name_match = re.match(r'^([^(\[]+)', line)
                            if name_match:
                                name = name_match.group(1).strip()
                                name = re.sub(r'\[.*?\]', '', name)
                                name = name.strip(' ,')
                                if name and len(name) > 2:
                                    ceos.append(name)
    return ', '.join(ceos) if ceos else None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index

for idx in indices:
    company = ceos.loc[idx, "company"]

    # skip if we already have a ceo
    if pd.notna(ceos.loc[idx, "CEO"]) and str(ceos.loc[idx, "CEO"]).strip():
        continue

    wiki_url = ceos.loc[idx, "wikipedia_page"]
    if pd.isna(wiki_url) or not str(wiki_url).strip():
        print(f"✗ {company}: no wikipedia_page link available")
        continue

    try:
        response = requests.get(wiki_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for {wiki_url}")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        ceo = extract_ceo_from_infobox(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing page - {str(e)[:80]}")
        continue

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        print(f"✓ {company}: found CEO {ceo}")
    else:
        print(f"✗ {company}: CEO not found in infobox")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✓ Visa: found CEO Ryan McInerney
✓ Travelers: found CEO Alan D. Schnitzer
✓ Amazon: found CEO Jeff Bezos
✓ Target: found CEO Brian Cornell
✗ Delta Air Lines: CEO not found in infobox
✓ Tesla: found CEO Elon Musk
✓ Coca-Cola: found CEO James Quincey
✗ Norfolk Southern: CEO not found in infobox
✓ Nike: found CEO Philip Knight
✗ General Motors: CEO not found in infobox
✓ Chevron: found CEO Mike Wirth
✓ Danaher: found CEO Steven Rales
✓ Alphabet: found CEO Sundar Pichai
✓ Southern: found CEO Chris Womack
✓ Apple: found CEO Arthur Levinson
✓ Walt Disney: found CEO Josh D'Amaro
✓ UPS: found CEO Carol B. Tomé
✓ RTX: found CEO Christopher T. Calio
✓ Progressive: found CEO Lawton W. Fitt
✓ Caterpillar: found CEO Joe Creed
✓ Eli Lilly: found CEO David A. Ricks
✓ Merck: found CEO Robert M. Davis
✓ Nationwide: found CEO Jeffrey Zellers
✓ Galaxy Digital: found CEO Michael Novogratz
✗ United Airlines Holdings: CEO not found in infobox
✗ Oracle: CEO not found in infobox
✓ HP: found CEO Chip Bergh
✓ I

#### Correction of CEOs with more accurate scraping

In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def replace_br_with_newline(tag):
    for br in tag.find_all("br"):
        br.replace_with("\n")


def extract_ceo_from_infobox(soup):
    """Extract CEO name from Wikipedia infobox, handling different 'Key people' layouts"""
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    for row in infobox.find_all('tr'):
        header = row.find('th')
        if not header:
            continue
        header_text = header.get_text().strip().lower()
        if 'key people' not in header_text and 'key person' not in header_text:
            continue

        data = row.find('td')
        if not data:
            continue

        # build a list of text segments: one per <li>, or split by <br> if no list
        lis = data.find_all('li')
        if lis:
            segments = [li.get_text(" ", strip=True) for li in lis]
        else:
            replace_br_with_newline(data)
            segments = [s.strip() for s in data.get_text().split('\n') if s.strip()]

        # merge segments that are only a parenthetical title into the previous segment
        merged = []
        for seg in segments:
            if seg.startswith('(') and merged:
                merged[-1] = merged[-1] + ' ' + seg
            else:
                merged.append(seg)

        # find segments mentioning CEO
        ceo_names = []
        for seg in merged:
            if re.search(r'\bCEO\b', seg, re.IGNORECASE) or re.search(r'chief executive officer', seg, re.IGNORECASE):
                name_match = re.match(r'^([^(]+)', seg)
                if name_match:
                    name = re.sub(r'\[.*?\]', '', name_match.group(1)).strip(' ,')
                    if name and len(name) > 2:
                        ceo_names.append(name)

        return ', '.join(ceo_names) if ceo_names else None

    return None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    wiki_url = ceos.loc[idx, "wikipedia_page"]

    if pd.isna(wiki_url) or not str(wiki_url).strip():
        continue

    try:
        response = requests.get(wiki_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for {wiki_url}")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        ceo = extract_ceo_from_infobox(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing page - {str(e)[:80]}")
        continue

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        print(f"✓ {company}: found CEO {ceo}")
    else:
        print(f"✗ {company}: CEO not found in infobox")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✓ Amazon: found CEO Andy Jassy
✓ Walmart: found CEO John Furner
✓ UnitedHealth Group: found CEO Stephen J. Hemsley, Tim Noel
✓ Apple: found CEO Tim Cook, John Ternus
✓ Alphabet: found CEO Sundar Pichai
✓ CVS Health: found CEO David Joyner
✓ Berkshire Hathaway: found CEO Greg Abel
✓ McKesson: found CEO Brian S. Tyler
✓ ExxonMobil Holdings: found CEO Darren Woods
✓ Cencora: found CEO Robert Mauch
✓ Microsoft: found CEO Satya Nadella
✓ JPMorgan Chase: found CEO Jamie Dimon
✓ Costco Wholesale: found CEO Ron Vachris
✓ Cigna Group: found CEO Brian Evanko
✓ Cardinal Health: found CEO Jason Hollar, Deborah Weitzman, Stephen Mason
✓ Nvidia: found CEO Jensen Huang
✓ Meta Platforms: found CEO Mark Zuckerberg
✓ Elevance Health: found CEO Gail Koziara Boudreaux
✓ Centene: found CEO Sarah London
✓ Bank of America: found CEO Brian Moynihan
✓ Chevron: found CEO Mike Wirth
✓ Ford Motor: found CEO Jim Farley
✓ General Motors: found CEO Alfred P. Sloan CEO
✓ Citigroup: found CEO Jane Fraser
✓ Home Depot:

### 2025

#### Getting the Wikipedia URLs with "oldid"

In [13]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from datetime import datetime
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

TARGET_DATE = datetime(2025, 7, 15)


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from a /wiki/Title url"""
    path = urlparse(url).path
    if path.startswith("/wiki/"):
        return path[len("/wiki/"):]
    return None


def find_closest_2025_oldid(soup):
    """Look at the history page and return the oldid of the revision closest to July 2025"""
    best_oldid = None
    best_diff = None

    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        if dt.year != 2025:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        diff = abs((dt - TARGET_DATE).days)
        if best_diff is None or diff < best_diff:
            best_diff = diff
            best_oldid = match.group(1)

    return best_oldid


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from year 2026
mask_2026 = ceos["year"] == 2026
wiki_2026_lookup = {}
for _, row in ceos[mask_2026].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_2026_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

mask_2025 = ceos["year"] == 2025
indices = ceos[mask_2025].index

for idx in indices:
    company = ceos.loc[idx, "company"]

    # reuse the 2026 wikipedia page if we already know it, otherwise guess it
    base_url = wiki_2026_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"✗ {company}: could not determine page title from {base_url}")
        continue

    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"

    try:
        response = requests.get(history_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for history page")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        oldid = find_closest_2025_oldid(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing history page - {str(e)[:80]}")
        continue

    if not oldid:
        print(f"✗ {company}: no 2025 revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"✓ {company}: {final_url}")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✓ Walmart: https://en.wikipedia.org/w/index.php?title=Walmart&oldid=1299843474
✓ Amazon: https://en.wikipedia.org/w/index.php?title=Amazon_(company)&oldid=1300977066
✓ UnitedHealth Group: https://en.wikipedia.org/w/index.php?title=UnitedHealth_Group&oldid=1300241872
✓ Apple: https://en.wikipedia.org/w/index.php?title=Apple_Inc.&oldid=1316476658
✓ CVS Health: https://en.wikipedia.org/w/index.php?title=CVS_Health&oldid=1300727937
✓ Berkshire Hathaway: https://en.wikipedia.org/w/index.php?title=Berkshire_Hathaway&oldid=1299445948
✓ Alphabet: https://en.wikipedia.org/w/index.php?title=Alphabet_Inc.&oldid=1300210973
✗ ExxonMobil Holdings: no 2025 revision found in history
✗ McKesson: no 2025 revision found in history
✓ Cencora: https://en.wikipedia.org/w/index.php?title=Cencora&oldid=1301265701
✓ JPMorgan Chase: https://en.wikipedia.org/w/index.php?title=JPMorgan_Chase&oldid=1300563182
✗ Costco Wholesale: no 2025 revision found in history
✗ Cigna Group: no 2025 revision found in history
✓ M

#### Improved scraping of "July 2025" pages

In [14]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from datetime import datetime
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

TARGET_DATE = datetime(2025, 7, 15)


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from a /wiki/Title url"""
    path = urlparse(url).path
    if path.startswith("/wiki/"):
        return path[len("/wiki/"):]
    return None


def find_closest_2025_oldid(soup):
    """Look at the history page and return the oldid of the revision closest to July 2025"""
    best_oldid = None
    best_diff = None

    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        date_text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff]', '', date_text)
        date_text = re.sub(r'\s+', ' ', date_text).strip()
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        if dt.year != 2025:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        diff = abs((dt - TARGET_DATE).days)
        if best_diff is None or diff < best_diff:
            best_diff = diff
            best_oldid = match.group(1)

    return best_oldid


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from year 2026
mask_2026 = ceos["year"] == 2026
wiki_2026_lookup = {}
for _, row in ceos[mask_2026].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_2026_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

# only look at 2025 rows that are still missing a wikipedia_page
mask_2025 = ceos["year"] == 2025
mask_missing = ceos["wikipedia_page"].isna() | (ceos["wikipedia_page"].astype(str).str.strip() == "")
indices = ceos[mask_2025 & mask_missing].index

for idx in indices:
    company = ceos.loc[idx, "company"]

    base_url = wiki_2026_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"✗ {company}: could not determine page title from {base_url}")
        continue

    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"

    try:
        response = requests.get(history_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for history page")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        oldid = find_closest_2025_oldid(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing history page - {str(e)[:80]}")
        continue

    if not oldid:
        print(f"✗ {company}: no 2025 revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"✓ {company}: {final_url}")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✗ ExxonMobil Holdings: no 2025 revision found in history
✗ McKesson: no 2025 revision found in history
✗ Costco Wholesale: no 2025 revision found in history
✗ Cigna Group: no 2025 revision found in history
✗ Ford Motor: no 2025 revision found in history
✗ Centene: no 2025 revision found in history
✗ Verizon Communications: no 2025 revision found in history
✗ Goldman Sachs Group: no 2025 revision found in history
✗ State Farm Insurance: no 2025 revision found in history
✗ StoneX Group: no 2025 revision found in history
✗ Procter & Gamble: no 2025 revision found in history
✗ New York Life Insurance: no 2025 revision found in history
✗ Publix Super Markets: no 2025 revision found in history
✗ Enterprise Products Partners: no 2025 revision found in history
✗ Capital One Financial: no 2025 revision found in history
✗ Cisco Systems: no 2025 revision found in history
✗ Liberty Mutual Insurance Group: no 2025 revision found in history
✗ Plains GP Holdings: no 2025 revision found in history
✗ B

### 2024

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urljoin
from datetime import datetime
import re
import time

# ---- change these two lines to reuse this script for a different year ----
YEAR = 2024
TARGET_DATE = datetime(2024, 7, 15)
# ----------------------------------------------------------------------------

LOOKUP_YEAR = 2026  # reuse the wikipedia page we already found for the next year
MAX_PAGES = 20          # safety limit on how many "older 500" pages we'll follow

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from either a /wiki/Title url or an index.php?title=Title url"""
    parsed = urlparse(url)
    if parsed.path.startswith("/wiki/"):
        return parsed.path[len("/wiki/"):]
    qs = parse_qs(parsed.query)
    if "title" in qs:
        return qs["title"][0]
    return None


def parse_revisions(soup):
    """Return a list of (datetime, oldid) for every revision link on the page"""
    revisions = []
    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        # strip invisible bidi/control characters MediaWiki inserts (LRM, RLM, etc.)
        date_text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff]', '', date_text)
        date_text = re.sub(r'\s+', ' ', date_text).strip()
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        revisions.append((dt, match.group(1)))
    return revisions


def find_next_page_url(soup, current_url):
    """Find the 'older 500' link, if any"""
    next_link = soup.find('a', {'rel': 'next'})
    if not next_link:
        return None
    return urljoin(current_url, next_link['href'])


def find_oldid_for_year(title, year, target_date):
    """Walk through history pages (following 'older 500') until we find revisions from `year`"""
    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"
    candidates = []

    for _ in range(MAX_PAGES):
        try:
            response = requests.get(history_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"    ✗ request failed - {str(e)[:80]}")
            return None

        if response.status_code != 200:
            print(f"    ✗ HTTP {response.status_code} for history page")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        revisions = parse_revisions(soup)

        for dt, oldid in revisions:
            if dt.year == year:
                candidates.append((dt, oldid))

        # once the oldest revision on this page is older than our target year,
        # there is no point paging further back
        if revisions and revisions[-1][0].year < year:
            break

        next_url = find_next_page_url(soup, history_url)
        if not next_url:
            break

        history_url = next_url
        time.sleep(1)

    if not candidates:
        return None

    best = min(candidates, key=lambda c: abs((c[0] - target_date).days))
    return best[1]


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from the next year already processed
mask_lookup = ceos["year"] == LOOKUP_YEAR
wiki_lookup = {}
for _, row in ceos[mask_lookup].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

mask_year = ceos["year"] == YEAR
mask_missing = ceos["wikipedia_page"].isna() | (ceos["wikipedia_page"].astype(str).str.strip() == "")
indices = ceos[mask_year & mask_missing].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    print(f"Processing: {company}")

    base_url = wiki_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"  ✗ could not determine page title from {base_url}")
        continue

    oldid = find_oldid_for_year(title, YEAR, TARGET_DATE)

    if not oldid:
        print(f"  ✗ no {YEAR} revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"  ✓ {final_url}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")

Processing: Walmart
  ✓ https://en.wikipedia.org/w/index.php?title=Walmart&oldid=1234339124
Processing: Amazon
  ✓ https://en.wikipedia.org/w/index.php?title=Amazon_(company)&oldid=1234715878
Processing: Apple
  ✓ https://en.wikipedia.org/w/index.php?title=Apple_Inc.&oldid=1234551441
Processing: UnitedHealth Group
  ✓ https://en.wikipedia.org/w/index.php?title=UnitedHealth_Group&oldid=1233962302
Processing: Berkshire Hathaway
  ✓ https://en.wikipedia.org/w/index.php?title=Berkshire_Hathaway&oldid=1233982074
Processing: CVS Health
  ✓ https://en.wikipedia.org/w/index.php?title=CVS_Health&oldid=1235402414
Processing: ExxonMobil Holdings
  ✗ no 2024 revision found in history
Processing: Alphabet
  ✓ https://en.wikipedia.org/w/index.php?title=Alphabet_Inc.&oldid=1234089317
Processing: McKesson
  ✗ no 2024 revision found in history
Processing: Cencora
  ✓ https://en.wikipedia.org/w/index.php?title=Cencora&oldid=1233194386
Processing: Costco Wholesale
  ✗ no 2024 revision found in history
Pr

### 2023

In [ ]:
# skip for now, as we already have the data from a different source

### 2022

In [18]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urljoin
from datetime import datetime
import re
import time

# ---- change these two lines to reuse this script for a different year ----
YEAR = 2022
TARGET_DATE = datetime(2022, 7, 15)
# ----------------------------------------------------------------------------

LOOKUP_YEAR = 2026  # reuse the wikipedia page we already found for the next year
MAX_PAGES = 20          # safety limit on how many "older 500" pages we'll follow

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from either a /wiki/Title url or an index.php?title=Title url"""
    parsed = urlparse(url)
    if parsed.path.startswith("/wiki/"):
        return parsed.path[len("/wiki/"):]
    qs = parse_qs(parsed.query)
    if "title" in qs:
        return qs["title"][0]
    return None


def parse_revisions(soup):
    """Return a list of (datetime, oldid) for every revision link on the page"""
    revisions = []
    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        # strip invisible bidi/control characters MediaWiki inserts (LRM, RLM, etc.)
        date_text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff]', '', date_text)
        date_text = re.sub(r'\s+', ' ', date_text).strip()
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        revisions.append((dt, match.group(1)))
    return revisions


def find_next_page_url(soup, current_url):
    """Find the 'older 500' link, if any"""
    next_link = soup.find('a', {'rel': 'next'})
    if not next_link:
        return None
    return urljoin(current_url, next_link['href'])


def find_oldid_for_year(title, year, target_date):
    """Walk through history pages (following 'older 500') until we find revisions from `year`"""
    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"
    candidates = []

    for _ in range(MAX_PAGES):
        try:
            response = requests.get(history_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"    ✗ request failed - {str(e)[:80]}")
            return None

        if response.status_code != 200:
            print(f"    ✗ HTTP {response.status_code} for history page")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        revisions = parse_revisions(soup)

        for dt, oldid in revisions:
            if dt.year == year:
                candidates.append((dt, oldid))

        # once the oldest revision on this page is older than our target year,
        # there is no point paging further back
        if revisions and revisions[-1][0].year < year:
            break

        next_url = find_next_page_url(soup, history_url)
        if not next_url:
            break

        history_url = next_url
        time.sleep(1)

    if not candidates:
        return None

    best = min(candidates, key=lambda c: abs((c[0] - target_date).days))
    return best[1]


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from the next year already processed
mask_lookup = ceos["year"] == LOOKUP_YEAR
wiki_lookup = {}
for _, row in ceos[mask_lookup].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

mask_year = ceos["year"] == YEAR
mask_missing = ceos["wikipedia_page"].isna() | (ceos["wikipedia_page"].astype(str).str.strip() == "")
indices = ceos[mask_year & mask_missing].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    print(f"Processing: {company}")

    base_url = wiki_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"  ✗ could not determine page title from {base_url}")
        continue

    oldid = find_oldid_for_year(title, YEAR, TARGET_DATE)

    if not oldid:
        print(f"  ✗ no {YEAR} revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"  ✓ {final_url}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")

Processing: Walmart
  ✓ https://en.wikipedia.org/w/index.php?title=Walmart&oldid=1098372309
Processing: Amazon
  ✓ https://en.wikipedia.org/w/index.php?title=Amazon_(company)&oldid=1099462106
Processing: Apple
  ✓ https://en.wikipedia.org/w/index.php?title=Apple_Inc.&oldid=1098855300
Processing: CVS Health
  ✓ https://en.wikipedia.org/w/index.php?title=CVS_Health&oldid=1097287458
Processing: UnitedHealth Group
  ✓ https://en.wikipedia.org/w/index.php?title=UnitedHealth_Group&oldid=1098025984
Processing: ExxonMobil Holdings
  ✗ no 2022 revision found in history
Processing: Berkshire Hathaway
  ✓ https://en.wikipedia.org/w/index.php?title=Berkshire_Hathaway&oldid=1099291637
Processing: Alphabet
  ✓ https://en.wikipedia.org/w/index.php?title=Alphabet_Inc.&oldid=1097870632
Processing: McKesson
  ✗ no 2022 revision found in history
Processing: Cencora
  ✓ https://en.wikipedia.org/w/index.php?title=Cencora&oldid=1107999920
Processing: Costco Wholesale
  ✗ no 2022 revision found in history
Pr

### 2021

In [19]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urljoin
from datetime import datetime
import re
import time

# ---- change these two lines to reuse this script for a different year ----
YEAR = 2021
TARGET_DATE = datetime(2021, 7, 15)
# ----------------------------------------------------------------------------

LOOKUP_YEAR = 2026  # reuse the wikipedia page we already found for the next year
MAX_PAGES = 20          # safety limit on how many "older 500" pages we'll follow

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from either a /wiki/Title url or an index.php?title=Title url"""
    parsed = urlparse(url)
    if parsed.path.startswith("/wiki/"):
        return parsed.path[len("/wiki/"):]
    qs = parse_qs(parsed.query)
    if "title" in qs:
        return qs["title"][0]
    return None


def parse_revisions(soup):
    """Return a list of (datetime, oldid) for every revision link on the page"""
    revisions = []
    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        # strip invisible bidi/control characters MediaWiki inserts (LRM, RLM, etc.)
        date_text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff]', '', date_text)
        date_text = re.sub(r'\s+', ' ', date_text).strip()
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        revisions.append((dt, match.group(1)))
    return revisions


def find_next_page_url(soup, current_url):
    """Find the 'older 500' link, if any"""
    next_link = soup.find('a', {'rel': 'next'})
    if not next_link:
        return None
    return urljoin(current_url, next_link['href'])


def find_oldid_for_year(title, year, target_date):
    """Walk through history pages (following 'older 500') until we find revisions from `year`"""
    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"
    candidates = []

    for _ in range(MAX_PAGES):
        try:
            response = requests.get(history_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"    ✗ request failed - {str(e)[:80]}")
            return None

        if response.status_code != 200:
            print(f"    ✗ HTTP {response.status_code} for history page")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        revisions = parse_revisions(soup)

        for dt, oldid in revisions:
            if dt.year == year:
                candidates.append((dt, oldid))

        # once the oldest revision on this page is older than our target year,
        # there is no point paging further back
        if revisions and revisions[-1][0].year < year:
            break

        next_url = find_next_page_url(soup, history_url)
        if not next_url:
            break

        history_url = next_url
        time.sleep(1)

    if not candidates:
        return None

    best = min(candidates, key=lambda c: abs((c[0] - target_date).days))
    return best[1]


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from the next year already processed
mask_lookup = ceos["year"] == LOOKUP_YEAR
wiki_lookup = {}
for _, row in ceos[mask_lookup].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

mask_year = ceos["year"] == YEAR
mask_missing = ceos["wikipedia_page"].isna() | (ceos["wikipedia_page"].astype(str).str.strip() == "")
indices = ceos[mask_year & mask_missing].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    print(f"Processing: {company}")

    base_url = wiki_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"  ✗ could not determine page title from {base_url}")
        continue

    oldid = find_oldid_for_year(title, YEAR, TARGET_DATE)

    if not oldid:
        print(f"  ✗ no {YEAR} revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"  ✓ {final_url}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")

Processing: Walmart
  ✓ https://en.wikipedia.org/w/index.php?title=Walmart&oldid=1033746848
Processing: Amazon
  ✓ https://en.wikipedia.org/w/index.php?title=Amazon_(company)&oldid=1033861573
Processing: Apple
  ✓ https://en.wikipedia.org/w/index.php?title=Apple_Inc.&oldid=1033806397
Processing: CVS Health
  ✓ https://en.wikipedia.org/w/index.php?title=CVS_Health&oldid=1033631837
Processing: UnitedHealth Group
  ✓ https://en.wikipedia.org/w/index.php?title=UnitedHealth_Group&oldid=1033439596
Processing: Berkshire Hathaway
  ✓ https://en.wikipedia.org/w/index.php?title=Berkshire_Hathaway&oldid=1033912085
Processing: McKesson
  ✗ no 2021 revision found in history
Processing: Cencora
  ✓ https://en.wikipedia.org/w/index.php?title=Cencora&oldid=1036468863
Processing: Alphabet
  ✓ https://en.wikipedia.org/w/index.php?title=Alphabet_Inc.&oldid=1033718086
Processing: ExxonMobil Holdings
  ✗ no 2021 revision found in history
Processing: AT&T
  ✓ https://en.wikipedia.org/w/index.php?title=AT&T&

### 2020

In [20]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urljoin
from datetime import datetime
import re
import time

# ---- change these two lines to reuse this script for a different year ----
YEAR = 2020
TARGET_DATE = datetime(2020, 7, 15)
# ----------------------------------------------------------------------------

LOOKUP_YEAR = 2026  # reuse the wikipedia page we already found for the next year
MAX_PAGES = 20          # safety limit on how many "older 500" pages we'll follow

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Guess a Wikipedia URL from a company name"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def get_title_from_url(url):
    """Extract the wiki page title from either a /wiki/Title url or an index.php?title=Title url"""
    parsed = urlparse(url)
    if parsed.path.startswith("/wiki/"):
        return parsed.path[len("/wiki/"):]
    qs = parse_qs(parsed.query)
    if "title" in qs:
        return qs["title"][0]
    return None


def parse_revisions(soup):
    """Return a list of (datetime, oldid) for every revision link on the page"""
    revisions = []
    for a in soup.find_all('a', class_='mw-changeslist-date'):
        date_text = a.get_text(strip=True)
        # strip invisible bidi/control characters MediaWiki inserts (LRM, RLM, etc.)
        date_text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff]', '', date_text)
        date_text = re.sub(r'\s+', ' ', date_text).strip()
        try:
            dt = datetime.strptime(date_text, "%H:%M, %d %B %Y")
        except ValueError:
            continue

        match = re.search(r'oldid=(\d+)', a.get('href', ''))
        if not match:
            continue

        revisions.append((dt, match.group(1)))
    return revisions


def find_next_page_url(soup, current_url):
    """Find the 'older 500' link, if any"""
    next_link = soup.find('a', {'rel': 'next'})
    if not next_link:
        return None
    return urljoin(current_url, next_link['href'])


def find_oldid_for_year(title, year, target_date):
    """Walk through history pages (following 'older 500') until we find revisions from `year`"""
    history_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=history&offset=&limit=500"
    candidates = []

    for _ in range(MAX_PAGES):
        try:
            response = requests.get(history_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"    ✗ request failed - {str(e)[:80]}")
            return None

        if response.status_code != 200:
            print(f"    ✗ HTTP {response.status_code} for history page")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        revisions = parse_revisions(soup)

        for dt, oldid in revisions:
            if dt.year == year:
                candidates.append((dt, oldid))

        # once the oldest revision on this page is older than our target year,
        # there is no point paging further back
        if revisions and revisions[-1][0].year < year:
            break

        next_url = find_next_page_url(soup, history_url)
        if not next_url:
            break

        history_url = next_url
        time.sleep(1)

    if not candidates:
        return None

    best = min(candidates, key=lambda c: abs((c[0] - target_date).days))
    return best[1]


ceos = pd.read_csv("data/ceos.csv")
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

# build lookup: lowercase company name -> wikipedia url, from the next year already processed
mask_lookup = ceos["year"] == LOOKUP_YEAR
wiki_lookup = {}
for _, row in ceos[mask_lookup].iterrows():
    if pd.notna(row["wikipedia_page"]) and str(row["wikipedia_page"]).strip():
        wiki_lookup[str(row["company"]).lower()] = row["wikipedia_page"]

mask_year = ceos["year"] == YEAR
mask_missing = ceos["wikipedia_page"].isna() | (ceos["wikipedia_page"].astype(str).str.strip() == "")
indices = ceos[mask_year & mask_missing].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    print(f"Processing: {company}")

    base_url = wiki_lookup.get(str(company).lower())
    if not base_url:
        base_url = get_wikipedia_url(company)

    title = get_title_from_url(base_url)
    if not title:
        print(f"  ✗ could not determine page title from {base_url}")
        continue

    oldid = find_oldid_for_year(title, YEAR, TARGET_DATE)

    if not oldid:
        print(f"  ✗ no {YEAR} revision found in history")
        continue

    final_url = f"https://en.wikipedia.org/w/index.php?title={title}&oldid={oldid}"
    ceos.loc[idx, "wikipedia_page"] = final_url
    print(f"  ✓ {final_url}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")

Processing: Walmart
  ✓ https://en.wikipedia.org/w/index.php?title=Walmart&oldid=967830216
Processing: Amazon
  ✓ https://en.wikipedia.org/w/index.php?title=Amazon_(company)&oldid=967496535
Processing: ExxonMobil Holdings
  ✗ no 2020 revision found in history
Processing: Apple
  ✓ https://en.wikipedia.org/w/index.php?title=Apple_Inc.&oldid=968056906
Processing: CVS Health
  ✓ https://en.wikipedia.org/w/index.php?title=CVS_Health&oldid=968789840
Processing: Berkshire Hathaway
  ✓ https://en.wikipedia.org/w/index.php?title=Berkshire_Hathaway&oldid=968462449
Processing: UnitedHealth Group
  ✓ https://en.wikipedia.org/w/index.php?title=UnitedHealth_Group&oldid=968200072
Processing: McKesson
  ✗ no 2020 revision found in history
Processing: AT&T
  ✓ https://en.wikipedia.org/w/index.php?title=AT&T&oldid=974587424
Processing: Cencora
  ✓ https://en.wikipedia.org/w/index.php?title=Cencora&oldid=968162562
Processing: Alphabet
  ✓ https://en.wikipedia.org/w/index.php?title=Alphabet_Inc.&oldid=96

### 2025-2020 CEOs from scraped Wikipedia URLs

In [21]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def replace_br_with_newline(tag):
    for br in tag.find_all("br"):
        br.replace_with("\n")


def extract_ceo_from_infobox(soup):
    """Extract CEO name from Wikipedia infobox, handling different 'Key people' layouts"""
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    for row in infobox.find_all('tr'):
        header = row.find('th')
        if not header:
            continue
        header_text = header.get_text().strip().lower()
        if 'key people' not in header_text and 'key person' not in header_text:
            continue

        data = row.find('td')
        if not data:
            continue

        # build a list of text segments: one per <li>, or split by <br> if no list
        lis = data.find_all('li')
        if lis:
            segments = [li.get_text(" ", strip=True) for li in lis]
        else:
            replace_br_with_newline(data)
            segments = [s.strip() for s in data.get_text().split('\n') if s.strip()]

        # merge segments that are only a parenthetical title into the previous segment
        merged = []
        for seg in segments:
            if seg.startswith('(') and merged:
                merged[-1] = merged[-1] + ' ' + seg
            else:
                merged.append(seg)

        # find segments mentioning CEO
        ceo_names = []
        for seg in merged:
            if re.search(r'\bCEO\b', seg, re.IGNORECASE) or re.search(r'chief executive officer', seg, re.IGNORECASE):
                name_match = re.match(r'^([^(]+)', seg)
                if name_match:
                    name = re.sub(r'\[.*?\]', '', name_match.group(1)).strip(' ,')
                    if name and len(name) > 2:
                        ceo_names.append(name)

        return ', '.join(ceo_names) if ceo_names else None

    return None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

mask_years = ceos["year"].between(2020, 2025)
mask_missing_ceo = ceos["CEO"].isna() | (ceos["CEO"].astype(str).str.strip() == "")
indices = ceos[mask_years & mask_missing_ceo].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    year = ceos.loc[idx, "year"]
    wiki_url = ceos.loc[idx, "wikipedia_page"]

    if pd.isna(wiki_url) or not str(wiki_url).strip():
        continue

    try:
        response = requests.get(wiki_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {year} {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {year} {company}: HTTP {response.status_code} for {wiki_url}")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        ceo = extract_ceo_from_infobox(soup)
    except Exception as e:
        print(f"✗ {year} {company}: error parsing page - {str(e)[:80]}")
        continue

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        print(f"✓ {year} {company}: found CEO {ceo}")
    else:
        print(f"✗ {year} {company}: CEO not found in infobox")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")

✓ 2025 Walmart: found CEO Doug McMillon
✓ 2025 Amazon: found CEO Andy Jassy
✓ 2025 UnitedHealth Group: found CEO Stephen J. Hemsley, Tim Noel
✗ 2025 Apple: request failed - HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read ti
✓ 2025 CVS Health: found CEO David Joyner
✓ 2025 Berkshire Hathaway: found CEO Warren Buffett
✓ 2025 Alphabet: found CEO Sundar Pichai
✓ 2025 Cencora: found CEO Robert Mauch
✓ 2025 JPMorgan Chase: found CEO Jamie Dimon
✗ 2025 Microsoft: request failed - HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read ti
✓ 2025 Cardinal Health: found CEO Jason Hollar, Deborah Weitzman, Stephen Mason
✓ 2025 Chevron: found CEO Mike Wirth
✓ 2025 Bank of America: found CEO Brian Moynihan
✓ 2025 General Motors: found CEO Alfred P. Sloan CEO
✓ 2025 Elevance Health: found CEO Gail Koziara Boudreaux
✓ 2025 Citigroup: found CEO Jane Fraser
✓ 2025 Meta Platforms: found CEO Mark Zuckerberg
✓ 2025 Home Depot: found CEO Ted Decker
✓ 2025

## Inspect collected data (CEOs 2020-2025)

In [ ]:
import pandas as pd

ceos = pd.read_csv("data/ceos.csv")

subset = ceos[ceos["year"].between(2020, 2025)]

unique_ceos = subset["CEO"].nunique()
unique_companies = subset["company"].nunique()
missing_ceos = subset["CEO"].isna().sum()
total_rows = len(subset)

print(f"unique CEOs: {unique_ceos}")
print(f"unique companies: {unique_companies}")
print(f"missing CEOs: {missing_ceos}/{total_rows}")